In [1]:
# Cella 1: Setup dell'Ambiente, Sincronizzazione Drive e Installazione Dipendenze
import sys
import os
from google.colab import drive, userdata
from huggingface_hub import login

print("1. Montaggio di Google Drive...")
drive.mount("/content/drive")

# Definiamo il percorso della tua repository su Drive
base_dir = '/content/drive/MyDrive/xai-project5'

if os.path.exists(base_dir):
    os.chdir(base_dir)
    print(f"✅ Sincronizzato con successo! Cartella di lavoro attuale: {os.getcwd()}")
else:
    raise FileNotFoundError(f"⚠️ Errore: Il percorso {base_dir} non esiste su Drive. Verifica il nome.")

# Configurazione dei percorsi relativi interni alla struttura della cartella src
feat_dir = os.path.join(base_dir, 'src', 'results', '01_feature_extraction')
dict_dir = os.path.join(base_dir, 'src', 'results', '02_dictionary_creation')
sae_dir = os.path.join(base_dir, 'src', 'results', '03_sae_training')
save_dir = os.path.join(base_dir, 'src', 'results', '04_evaluation')
scripts_path = os.path.join(base_dir, 'src', 'scripts')

os.makedirs(save_dir, exist_ok=True)

if scripts_path not in sys.path:
    sys.path.append(scripts_path)

# 2. Recupero del token per MedGemma (Richiede approvazione su Hugging Face)
print("\n2. Recupero credenziali dai Secrets di Colab...")
HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Autenticazione Hugging Face completata con successo!")
else:
    raise ValueError("⚠️ HF_TOKEN non trovato nei tuoi Secrets di Colab. MedGemma richiede l'autenticazione.")

# 3. Installazione dei pacchetti richiesti per l'LLM locale
print("\n3. Installazione dei pacchetti per l'inferenza dell'LLM...")
!pip install -q transformers accelerate

print("\n🚀 Setup iniziale completato. Pronto per inizializzare il modello valutatore!")

1. Montaggio di Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Sincronizzato con successo! Cartella di lavoro attuale: /content/drive/MyDrive/xai-project5

2. Recupero credenziali dai Secrets di Colab...
✅ Autenticazione Hugging Face completata con successo!

3. Installazione dei pacchetti per l'inferenza dell'LLM...

🚀 Setup iniziale completato. Pronto per inizializzare il modello valutatore!


In [2]:
# Cella 2: Caricamento del Modello Valutatore MedGemma
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device rilevato per la valutazione: {device}")

model_id = "google/medgemma-1.5-4b-it"

model_eval = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,   # bfloat16, non float16, è quello raccomandato per Gemma3
    device_map="auto",
)
processor_eval = AutoProcessor.from_pretrained(model_id)
tokenizer_eval = processor_eval.tokenizer  # se ti serve ancora altrove

print("✅ MedGemma caricato e pronto per agire come Frozen External Evaluator!")

Device rilevato per la valutazione: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✅ MedGemma caricato e pronto per agire come Frozen External Evaluator!


In [15]:
import os
import torch
import torch.nn.functional as F
from sae import SparseAutoencoder

print("1. Caricamento dei Modelli e del Dizionario...")

dict_path = os.path.join(dict_dir, 'biomedclip_medical_dictionary_embeddings.pt')
dictionary_data = torch.load(dict_path, map_location=device)
medical_concepts = list(dictionary_data.keys())
T_matrix = torch.stack(list(dictionary_data.values())).to(device)
if T_matrix.dim() == 3:
    T_matrix = T_matrix.squeeze(1)

INPUT_DIM = 512
HIDDEN_DIM = 1024

sae = SparseAutoencoder(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
sae_path = os.path.join(sae_dir, 'sae_model_final_e300_l0.001_h1024.pt')
state_dict = torch.load(sae_path, map_location=device)

if isinstance(state_dict, dict) and 'state_dict' in state_dict:
    sae.load_state_dict(state_dict['state_dict'])
elif isinstance(state_dict, dict) and not hasattr(state_dict, 'eval'):
    sae.load_state_dict(state_dict)
else:
    sae = state_dict

sae.eval()

with torch.no_grad():
    sae_decoder_weights = F.normalize(sae.decoder.weight.data, p=2, dim=0)
    concept_similarities = torch.matmul(T_matrix, sae_decoder_weights)

print("2. Caricamento del Dataset e dei Referti...")
dataset_path = os.path.join(feat_dir, 'biomedclip_openi_with_reports.pt')
dataset_data = torch.load(dataset_path, map_location=device)

vision_embeddings = dataset_data['embeddings'].to(torch.float32)
reports = dataset_data.get('reports', [])

def evaluate_concept_with_llm(report, concept, model, processor, device):
    prompt_text = f"""You are a text-only medical NLP classifier. You are NOT given any image, and none is needed — this task is based purely on the written report text below.
Do not ask for clarification, do not mention missing information, do not request an image. Just classify.

Task: Determine if the Concept is Aligned, Unaligned, or Uncertain with respect to the Report text.
Reply with EXACTLY one word at the very end of your answer: Aligned, Unaligned, or Uncertain.

--- EXAMPLES ---
Report: "The cardiac silhouette and mediastinum size are within normal limits. Normal chest x-ray."
Concept: "airspace disease"
Verdict: Unaligned

Report: "Lungs are overall hyperexpanded with flattening of the diaphragms. Degenerative changes in the thoracic spine."
Concept: "bone diseases"
Verdict: Aligned

Report: "Heart size is normal. Lungs are clear bilaterally. No acute cardiopulmonary abnormality."
Concept: "aortic aneurysm"
Verdict: Unaligned

Report: "Opacity in the right lower lobe, cannot exclude pneumonia."
Concept: "blister"
Verdict: Uncertain
--- END OF EXAMPLES ---

Now evaluate this real case (text only, no image involved):
Report: "{report}"
Concept: "{concept}"
Verdict:"""

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt_text}]}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            min_new_tokens=3,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
        )
        generation = generation[0][input_len:]

    generated_text = processor.decode(generation, skip_special_tokens=True).strip()
    text_lower = generated_text.lower()

    candidates = []
    idx = text_lower.rfind("unaligned")
    if idx != -1: candidates.append((idx, "Unaligned"))
    idx = text_lower.rfind("uncertain")
    if idx != -1: candidates.append((idx, "Uncertain"))
    search_from = 0
    while True:
        idx = text_lower.find("aligned", search_from)
        if idx == -1:
            break
        if not (idx >= 2 and text_lower[idx-2:idx] == "un"):
            candidates.append((idx, "Aligned"))
        search_from = idx + 1

    verdict = max(candidates, key=lambda x: x[0])[1] if candidates else "Uncertain"

    print(f"[DEBUG] {concept!r} -> tail='...{generated_text[-100:]}' => {verdict}")
    return verdict
print("✅ Moduli di inferenza e allineamento semantico pronti.")

1. Caricamento dei Modelli e del Dizionario...
2. Caricamento del Dataset e dei Referti...
✅ Moduli di inferenza e allineamento semantico pronti.


In [17]:
import pandas as pd
import numpy as np
from tqdm import tqdm

TOP_K = 10
NUM_SAMPLES = 10

results = []

print(f"Inizio valutazione su un sottoinsieme di {NUM_SAMPLES} immagini usando MedGemma...\n")

for i in tqdm(range(min(NUM_SAMPLES, len(vision_embeddings)))):
    img_emb = vision_embeddings[i].unsqueeze(0).to(device)

    try:
        report = reports[i]
    except Exception:
        continue

    if not report or len(str(report).strip()) < 5:
        continue

    with torch.no_grad():
        _, z = sae(img_emb)

    activations = z[0]

    top_vals, top_indices = torch.topk(activations, TOP_K)
    active_neurons = top_indices[top_vals > 0.0]

    if len(active_neurons) == 0:
        continue

    predicted_concepts = set()

    for neuron_idx in active_neurons:
        neuron_concept_scores = concept_similarities[:, neuron_idx]
        best_concept_idx = torch.argmax(neuron_concept_scores).item()
        concept = medical_concepts[best_concept_idx]
        predicted_concepts.add(concept)

    predicted_concepts = list(predicted_concepts)

    aligned_count = 0
    unaligned_count = 0
    uncertain_count = 0
    concept_details = []

    for concept in predicted_concepts:
        verdict = evaluate_concept_with_llm(report, concept, model_eval, processor_eval, device)

        if verdict == "Aligned":
            aligned_count += 1
        elif verdict == "Unaligned":
            unaligned_count += 1
        else:
            uncertain_count += 1

        concept_details.append({"concept": concept, "verdict": verdict})

    total = len(predicted_concepts)

    if total > 0:
        results.append({
            "image_index": i,
            "report": report,
            "total_concepts_found": total,
            "aligned_score": aligned_count / total,
            "unaligned_score": unaligned_count / total,
            "uncertain_score": uncertain_count / total,
            "details": concept_details
        })

df_evaluation = pd.DataFrame(results)

print(f"\n✅ Valutazione completata! Analizzate con successo {len(df_evaluation)} immagini valide.")
display(df_evaluation.head())

Inizio valutazione su un sottoinsieme di 10 immagini usando MedGemma...



  0%|          | 0/10 [00:00<?, ?it/s][transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[DEBUG] 'airspace disease' -> tail='...Uncertain' => Uncertain


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[DEBUG] 'aorta' -> tail='...Uncertain' => Uncertain


...

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


[DEBUG] 'airspace disease' -> tail='... normally inflated out evidence about focal airspace sick, pleural effusionity, instead pneumothorax' => Uncertain


100%|██████████| 10/10 [08:42<00:00, 52.21s/it]

[DEBUG] 'bronchiectasis' -> tail='...within normal limits for si... [cut short]" - This part seems irrelevant to the core findings.
    >' => Uncertain

✅ Valutazione completata! Analizzate con successo 10 immagini valide.


,image_index,report,total_concepts_found,aligned_score,unaligned_score,uncertain_score,details
0,0,The cardiac silhouette and mediastinum size ar...,4,0.0,0.0,1.0,"[{'concept': 'airspace disease', 'verdict': 'U..."
1,1,Borderline cardiomegaly. Midline sternotomy XX...,7,0.0,0.0,1.0,"[{'concept': 'aorta', 'verdict': 'Uncertain'},..."
2,2,"No displaced rib fractures, pneumothorax, or p...",8,0.0,0.0,1.0,"[{'concept': 'aorta', 'verdict': 'Uncertain'},..."
3,3,There are diffuse bilateral interstitial and a...,5,0.2,0.0,0.8,"[{'concept': 'abdomen', 'verdict': 'Uncertain'..."
4,4,The cardiomediastinal silhouette and pulmonary...,5,0.0,0.0,1.0,"[{'concept': 'abdomen', 'verdict': 'Uncertain'..."


In [18]:
# Cella 5: Analisi Qualitativa (Best, Median, Worst Cases)
print("=== ANALISI MEDCONCEPT: RANKING DELLE SPIEGAZIONI ===\n")

# Filtriamo solo le immagini dove il SAE ha trovato almeno 1 concetto
valid_df = df_evaluation[df_evaluation['total_concepts_found'] > 0].copy()

# Ordiniamo il DataFrame in base all'Aligned Score (dal più alto al più basso)
sorted_df = valid_df.sort_values(by='aligned_score', ascending=False).reset_index(drop=True)

# Identifichiamo gli indici per Best, Median e Worst
best_idx = 0                                     # Il primo in classifica
median_idx = len(sorted_df) // 2                 # Quello esattamente a metà
worst_idx = len(sorted_df) - 1                   # L'ultimo in classifica

casi_studio = {
    "🏆 BEST CASE (Massimo Allineamento)": sorted_df.iloc[best_idx],
    "⚖️ MEDIAN CASE (Allineamento Parziale/Ambiguo)": sorted_df.iloc[median_idx],
    "🚨 WORST CASE (Allineamento Minimo/Rumore)": sorted_df.iloc[worst_idx]
}

for titolo, caso in casi_studio.items():
    print("-" * 60)
    print(f"{titolo}")
    print(f"Indice Immagine Originale: {caso['image_index']}")
    print(f"Punteggi -> Aligned: {caso['aligned_score']:.2f} | Unaligned: {caso['unaligned_score']:.2f} | Uncertain: {caso['uncertain_score']:.2f}")
    print(f"\nREFERTO ORIGINALE (Ground Truth):\n{caso['report']}\n")

    print("CONCETTI ESTRATTI DAL SAE E VERDETTO DI MEDGEMMA:")
    for detail in caso['details']:
        conc = detail['concept']
        verd = detail['verdict']

        # Aggiungiamo un'icona per leggibilità
        if verd == "Aligned": icon = "✅"
        elif verd == "Unaligned": icon = "❌"
        else: icon = "⚠️"

        print(f"  {icon} {conc:<25} -> {verd}")
    print("\n")

=== ANALISI MEDCONCEPT: RANKING DELLE SPIEGAZIONI ===

------------------------------------------------------------
🏆 BEST CASE (Massimo Allineamento)
Indice Immagine Originale: 3
Punteggi -> Aligned: 0.20 | Unaligned: 0.00 | Uncertain: 0.80

REFERTO ORIGINALE (Ground Truth):
There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the left lung apex, that could represent a cavitary lesion in the left lung apex.There are streaky opacities in the right upper lobe, XXXX scarring. The cardiomediastinal silhouette is normal in size and contour. There is no pneumothorax or large pleural effusion. 1. Bullous emphysema and interstitial fibrosis. 2. Probably scarring in the left apex, although difficult to exclude a cavitary lesion. 3. Opacities in the bilateral upper lobes could represent scarring, however the absence of comparison exam, recommend short interval followup radiograph 